
# Teacher Regularization v2.0

`baseline v2.1`를 기준 backbone으로 사용하고, `DANN v1.0`의 domain adaptation loss를 참고해 teacher regularization을 추가한 notebook입니다.

구성:
- student: baseline v2.1의 `MultiViewBidirectionalCrossAttention`
- teacher target group 1: `physics`
- teacher target group 2: `image_structure`
- auxiliary: `domain_head` with GRL


In [1]:

from __future__ import annotations

import os
import sys
import json
import random
import shutil
from dataclasses import dataclass
from itertools import cycle
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.autograd import Function
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.preprocessing import StandardScaler
import timm

SRC_DIR = (Path.cwd() / '../../src').resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from augmentations import build_default_transforms
from output_paths import allocate_output_paths
from reproducibility import make_generator, seed_everything, seed_worker
from models import (
    EMAConfig,
    ModelEMA,
    MultiViewBidirectionalCrossAttentionConfig,
)

DATA_DIR = (Path.cwd() / '../../data').resolve()
FEATURE_CSV = (Path.cwd() / '../../outputs/physics_feature_analysis_v2/class_analysis_features.csv').resolve()
assert DATA_DIR.exists(), f'data dir not found: {DATA_DIR}'
assert FEATURE_CSV.exists(), f'feature csv not found: {FEATURE_CSV}'
print('DATA_DIR:', DATA_DIR)
print('FEATURE_CSV:', FEATURE_CSV)

PHYSICS_FEATURES = [
    'top_fill_ratio',
    'top_centroid_dx',
    'front_centroid_dx',
    'front_tilt',
    'top_support_width_frac',
]
IMAGE_STRUCTURE_FEATURES = [
    'abs_delta_structure_center_x',
    'top_structure_center_x',
    'front_structure_center_x',
    'mean_structure_center_x',
    'mean_structure_bbox_w',
]
ALL_TEACHER_FEATURES = PHYSICS_FEATURES + IMAGE_STRUCTURE_FEATURES

CFG = {
    'IMG_SIZE': 224,
    'EPOCHS': 30,
    'LEARNING_RATE': 3e-4,
    'BATCH_SIZE': 16,
    'SEED': 42,
    'NUM_WORKERS': 8,
    'WEIGHT_DECAY': 1e-4,
    'MIN_LR': 1e-6,
    'EMA_DECAY': 0.999,
    'EMA_USE_FOR_EVAL': True,
    'EARLY_STOPPING_PATIENCE': 5,
    'MIXUP_ALPHA': 0.1,
    'MIXUP_PROB': 0.0,
    'VIDEO_AUG_ENABLE': True,
    'VIDEO_AUG_CACHE': True,
    'UNSTABLE_START_MIN_SEC': 0.5,
    'UNSTABLE_START_MAX_SEC': 1.0,
    'UNSTABLE_FRAMES_MIN': 2,
    'UNSTABLE_FRAMES_MAX': 3,
    'STABLE_END_MIN_SEC': 9.0,
    'STABLE_END_MAX_SEC': 10.0,
    'STABLE_FRAMES_PER_VIDEO': 2,
    'BACKBONE_NAME': 'swin_tiny_patch4_window7_224.ms_in22k_ft_in1k',
    'ATTN_DIM': 256,
    'NUM_HEADS': 8,
    'NUM_LAYERS': 2,
    'POS_GRID': 7,
    'DROPOUT': 0.1,
    'CLASSIFIER_HIDDEN_DIM': 512,
    'CLASSIFIER_MID_DIM': 128,
    'CLASSIFIER_DROPOUT': 0.3,
    'DOMAIN_HIDDEN_DIM': 256,
    'DOMAIN_DROPOUT': 0.2,
    'PHYSICS_LOSS_WEIGHT': 0.05,
    'IMAGE_LOSS_WEIGHT': 0.10,
    'DOMAIN_LOSS_WEIGHT': 0.05,
    'GRL_WARMUP_EPOCHS': 5,
    'GRL_MAX_LAMBDA': 0.2,
    'GRL_GAMMA': 4.0,
}

seed_everything(CFG['SEED'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


DATA_DIR: /media/hdd0/whyz/structure-stability/data
FEATURE_CSV: /media/hdd0/whyz/structure-stability/outputs/physics_feature_analysis_v2/class_analysis_features.csv
device: cuda


In [2]:

train_df = pd.read_csv(DATA_DIR / 'train.csv', encoding='utf-8-sig')
dev_df = pd.read_csv(DATA_DIR / 'dev.csv', encoding='utf-8-sig')
test_df = pd.read_csv(DATA_DIR / 'sample_submission.csv', encoding='utf-8-sig')
feature_df = pd.read_csv(FEATURE_CSV, encoding='utf-8-sig')

feature_df = feature_df[['split', 'sample_id'] + ALL_TEACHER_FEATURES].copy()
feature_df = feature_df.rename(columns={'sample_id': 'id'})

for split_name, split_df in [('train', train_df), ('dev', dev_df)]:
    part = feature_df[feature_df['split'].eq(split_name)].drop(columns=['split']).copy()
    missing = set(split_df['id']) - set(part['id'])
    print(split_name, 'teacher rows:', len(part), 'missing:', len(missing))

physics_scaler = StandardScaler()
image_scaler = StandardScaler()
train_phys_mask = feature_df['split'].eq('train') & feature_df[PHYSICS_FEATURES].notna().all(axis=1)
train_img_mask = feature_df['split'].eq('train') & feature_df[IMAGE_STRUCTURE_FEATURES].notna().all(axis=1)
physics_scaler.fit(feature_df.loc[train_phys_mask, PHYSICS_FEATURES])
image_scaler.fit(feature_df.loc[train_img_mask, IMAGE_STRUCTURE_FEATURES])

all_phys_mask = feature_df[PHYSICS_FEATURES].notna().all(axis=1)
all_img_mask = feature_df[IMAGE_STRUCTURE_FEATURES].notna().all(axis=1)
feature_df.loc[all_phys_mask, PHYSICS_FEATURES] = physics_scaler.transform(feature_df.loc[all_phys_mask, PHYSICS_FEATURES])
feature_df.loc[all_img_mask, IMAGE_STRUCTURE_FEATURES] = image_scaler.transform(feature_df.loc[all_img_mask, IMAGE_STRUCTURE_FEATURES])

print('teacher target normalization: StandardScaler fitted on train split')
print('physics normalized rows:', int(all_phys_mask.sum()), 'image normalized rows:', int(all_img_mask.sum()))

train_df = train_df.merge(feature_df[feature_df['split'].eq('train')].drop(columns=['split']), on='id', how='left')
dev_df = dev_df.merge(feature_df[feature_df['split'].eq('dev')].drop(columns=['split']), on='id', how='left')
test_df['sample_dir'] = str(DATA_DIR / 'test')

print('train:', train_df.shape)
print('dev:', dev_df.shape)


train teacher rows: 1000 missing: 0
dev teacher rows: 100 missing: 0
teacher target normalization: StandardScaler fitted on train split
physics normalized rows: 1100 image normalized rows: 1100
train: (1000, 12)
dev: (100, 12)


In [3]:

def _extract_frame_by_sec(cap, sec, fps, frame_count):
    frame_idx = int(max(0, min(frame_count - 1, round(sec * fps))))
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, frame = cap.read()
    if not ok:
        return None
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)


def _extract_last_frame(cap, frame_count):
    last_idx = max(0, frame_count - 1)
    cap.set(cv2.CAP_PROP_POS_FRAMES, last_idx)
    ok, frame = cap.read()
    if not ok:
        return None
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)


def _video_aug_cache_signature(cfg):
    keys = [
        'SEED',
        'UNSTABLE_START_MIN_SEC',
        'UNSTABLE_START_MAX_SEC',
        'UNSTABLE_FRAMES_MIN',
        'UNSTABLE_FRAMES_MAX',
        'STABLE_END_MIN_SEC',
        'STABLE_END_MAX_SEC',
        'STABLE_FRAMES_PER_VIDEO',
    ]
    return {k: cfg.get(k) for k in keys}


def build_video_augmented_df(train_df, data_dir, cfg):
    train_root = data_dir / 'train'
    aug_root = data_dir / 'train_video_aug'
    aug_root.mkdir(parents=True, exist_ok=True)

    cache_csv = aug_root / 'aug_df.csv'
    cache_meta = aug_root / 'cache_meta.json'
    cache_sig = _video_aug_cache_signature(cfg)

    if cfg.get('VIDEO_AUG_CACHE', True) and cache_csv.exists() and cache_meta.exists():
        try:
            meta = json.loads(cache_meta.read_text())
            if meta.get('signature') == cache_sig:
                cached_df = pd.read_csv(cache_csv)
                if not cached_df.empty:
                    cached_df['sample_dir'] = str(aug_root)
                    return cached_df
        except Exception as exc:
            print('video aug cache read failed:', exc)

    for p in aug_root.glob('AUGV_*'):
        if p.is_dir():
            shutil.rmtree(p, ignore_errors=True)

    rng = np.random.default_rng(cfg['SEED'])
    stable_rows = []
    unstable_rows = []
    saved_idx = 0

    def save_aug(img, label):
        nonlocal saved_idx
        aug_id = f'AUGV_{saved_idx:07d}'
        out_dir = aug_root / aug_id
        out_dir.mkdir(parents=True, exist_ok=True)
        Image.fromarray(img).save(out_dir / 'front.png')
        Image.fromarray(img).save(out_dir / 'top.png')
        row = {'id': aug_id, 'label': label, 'sample_dir': str(aug_root)}
        for col in ALL_TEACHER_FEATURES:
            row[col] = np.nan
        saved_idx += 1
        return row

    for sample_id in tqdm(train_df['id'].tolist(), desc='Video aug stable(last-frame)', dynamic_ncols=True, ascii=True):
        video_path = train_root / sample_id / 'simulation.mp4'
        if not video_path.exists():
            continue
        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            cap.release()
            continue
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if frame_count <= 0:
            cap.release()
            continue
        img = _extract_last_frame(cap, frame_count)
        cap.release()
        if img is not None:
            stable_rows.append(save_aug(img, 'stable'))

    unstable_ids = train_df.loc[train_df['label'] == 'unstable', 'id'].tolist()
    for sample_id in tqdm(unstable_ids, desc='Video aug unstable(0.5~1.0s)', dynamic_ncols=True, ascii=True):
        video_path = train_root / sample_id / 'simulation.mp4'
        if not video_path.exists():
            continue
        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            cap.release()
            continue
        fps = cap.get(cv2.CAP_PROP_FPS)
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if fps is None or fps <= 0 or frame_count <= 1:
            cap.release()
            continue
        duration = frame_count / fps
        low = cfg['UNSTABLE_START_MIN_SEC']
        high = min(cfg['UNSTABLE_START_MAX_SEC'], max(0.0, duration - 1.0 / fps))
        if high <= low:
            cap.release()
            continue
        n_unstable = int(rng.integers(cfg['UNSTABLE_FRAMES_MIN'], cfg['UNSTABLE_FRAMES_MAX'] + 1))
        unstable_secs = rng.uniform(low, high, size=n_unstable)
        for sec in unstable_secs:
            img = _extract_frame_by_sec(cap, float(sec), fps, frame_count)
            if img is not None:
                unstable_rows.append(save_aug(img, 'unstable'))
        cap.release()

    stable_df = pd.DataFrame(stable_rows)
    unstable_df = pd.DataFrame(unstable_rows)
    if stable_df.empty or unstable_df.empty:
        return pd.DataFrame(columns=['id', 'label', 'sample_dir'])

    target_unstable = len(stable_df)
    if len(unstable_df) >= target_unstable:
        unstable_bal = unstable_df.sample(n=target_unstable, random_state=cfg['SEED'])
    else:
        unstable_bal = unstable_df.sample(n=target_unstable, replace=True, random_state=cfg['SEED'])

    aug_df = pd.concat([stable_df, unstable_bal], ignore_index=True)
    aug_df = aug_df.sample(frac=1.0, random_state=cfg['SEED']).reset_index(drop=True)
    aug_df.to_csv(cache_csv, index=False)
    cache_meta.write_text(json.dumps({'signature': cache_sig}, ensure_ascii=False, indent=2))
    return aug_df


class TeacherRegularizationDataset(Dataset):
    def __init__(self, df, root_dir, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform
        self.is_test = is_test
        self.label_map = {'stable': 0, 'unstable': 1}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        sample_id = str(row['id'])
        base_dir = self.root_dir
        if ('sample_dir' in self.df.columns) and pd.notna(row.get('sample_dir', np.nan)):
            base_dir = str(row['sample_dir'])
        folder_path = os.path.join(base_dir, sample_id)

        views = []
        for name in ['front', 'top']:
            img_path = os.path.join(folder_path, f'{name}.png')
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            views.append(image)

        if self.is_test:
            return {'views': views, 'id': sample_id}

        label = self.label_map[row['label']]
        physics_target = row[PHYSICS_FEATURES].to_numpy(dtype=np.float32)
        image_target = row[IMAGE_STRUCTURE_FEATURES].to_numpy(dtype=np.float32)
        return {
            'views': views,
            'label': torch.tensor(label, dtype=torch.float32),
            'physics_target': torch.tensor(physics_target, dtype=torch.float32),
            'image_target': torch.tensor(image_target, dtype=torch.float32),
            'has_teacher': torch.tensor(float(np.isfinite(physics_target).all() and np.isfinite(image_target).all()), dtype=torch.float32),
            'id': sample_id,
        }


train_transform, test_transform = build_default_transforms(CFG['IMG_SIZE'])
train_df_for_train = train_df.copy()
train_df_for_train['sample_dir'] = str(DATA_DIR / 'train')
if CFG['VIDEO_AUG_ENABLE']:
    aug_df = build_video_augmented_df(train_df, DATA_DIR, CFG)
    if not aug_df.empty:
        train_df_for_train = pd.concat([train_df_for_train, aug_df], ignore_index=True)

dev_domain_df = dev_df.copy()
dev_domain_df['sample_dir'] = str(DATA_DIR / 'dev')
dev_eval_df = dev_df.copy()
dev_eval_df['sample_dir'] = str(DATA_DIR / 'dev')

train_dataset = TeacherRegularizationDataset(train_df_for_train, str(DATA_DIR / 'train'), train_transform, is_test=False)
dev_domain_dataset = TeacherRegularizationDataset(dev_domain_df, str(DATA_DIR / 'dev'), train_transform, is_test=False)
dev_eval_dataset = TeacherRegularizationDataset(dev_eval_df, str(DATA_DIR / 'dev'), test_transform, is_test=False)
test_dataset = TeacherRegularizationDataset(test_df, str(DATA_DIR / 'test'), test_transform, is_test=True)

loader_kwargs = dict(
    batch_size=CFG['BATCH_SIZE'],
    num_workers=CFG['NUM_WORKERS'],
    pin_memory=(device.type == 'cuda'),
)
train_loader = DataLoader(train_dataset, shuffle=True, worker_init_fn=seed_worker, generator=make_generator(CFG['SEED']), **loader_kwargs)
dev_domain_loader = DataLoader(dev_domain_dataset, shuffle=True, worker_init_fn=seed_worker, generator=make_generator(CFG['SEED'] + 1), **loader_kwargs)
dev_eval_loader = DataLoader(dev_eval_dataset, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)

print('train_dataset:', len(train_dataset))
print('dev_domain_dataset:', len(dev_domain_dataset))
print('dev_eval_dataset:', len(dev_eval_dataset))
print('test_dataset:', len(test_dataset))


train_dataset: 3000
dev_domain_dataset: 100
dev_eval_dataset: 100
test_dataset: 1000


In [4]:

@dataclass(frozen=True)
class TeacherRegularizationConfig:
    backbone_name: str = CFG['BACKBONE_NAME']
    pretrained: bool = True
    attn_dim: int = CFG['ATTN_DIM']
    num_heads: int = CFG['NUM_HEADS']
    num_layers: int = CFG['NUM_LAYERS']
    pos_grid: int = CFG['POS_GRID']
    dropout: float = CFG['DROPOUT']
    classifier_hidden_dim: int = CFG['CLASSIFIER_HIDDEN_DIM']
    classifier_mid_dim: int = CFG['CLASSIFIER_MID_DIM']
    classifier_dropout: float = CFG['CLASSIFIER_DROPOUT']
    domain_hidden_dim: int = CFG['DOMAIN_HIDDEN_DIM']
    domain_dropout: float = CFG['DOMAIN_DROPOUT']
    physics_dim: int = len(PHYSICS_FEATURES)
    image_dim: int = len(IMAGE_STRUCTURE_FEATURES)
    out_dim: int = 1


class GradientReversalFunction(Function):
    @staticmethod
    def forward(ctx, x, lambda_):
        ctx.lambda_ = lambda_
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambda_ * grad_output, None


def grad_reverse(x, lambda_=1.0):
    return GradientReversalFunction.apply(x, lambda_)


class CrossAttentionBlock(nn.Module):
    def __init__(self, dim: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        self.norm_q = nn.LayerNorm(dim)
        self.norm_kv = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)

    def forward(self, q_tokens, kv_tokens):
        q = self.norm_q(q_tokens)
        kv = self.norm_kv(kv_tokens)
        attn_out, _ = self.attn(q, kv, kv, need_weights=False)
        return q_tokens + attn_out


class TeacherRegularizedMultiView(nn.Module):
    def __init__(self, config: TeacherRegularizationConfig | None = None):
        super().__init__()
        self.config = config or TeacherRegularizationConfig()
        self.backbone = timm.create_model(
            self.config.backbone_name,
            pretrained=self.config.pretrained,
            num_classes=0,
            global_pool='',
        )
        self.feature_dim = int(getattr(self.backbone, 'num_features'))
        self.cnn_proj = nn.Conv2d(self.feature_dim, self.config.attn_dim, kernel_size=1, bias=False)
        self.token_proj = nn.Linear(self.feature_dim, self.config.attn_dim, bias=False)
        self.cross_12 = nn.ModuleList([CrossAttentionBlock(self.config.attn_dim, self.config.num_heads, self.config.dropout) for _ in range(self.config.num_layers)])
        self.cross_21 = nn.ModuleList([CrossAttentionBlock(self.config.attn_dim, self.config.num_heads, self.config.dropout) for _ in range(self.config.num_layers)])
        self.classifier = nn.Sequential(
            nn.Linear(self.config.attn_dim * 2, self.config.classifier_hidden_dim),
            nn.BatchNorm1d(self.config.classifier_hidden_dim),
            nn.ReLU(),
            nn.Dropout(self.config.classifier_dropout),
            nn.Linear(self.config.classifier_hidden_dim, self.config.classifier_mid_dim),
            nn.ReLU(),
            nn.Linear(self.config.classifier_mid_dim, self.config.out_dim),
        )
        self.physics_head = nn.Sequential(
            nn.Linear(self.config.attn_dim * 2, self.config.classifier_hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(self.config.classifier_hidden_dim // 2, self.config.physics_dim),
        )
        self.image_head = nn.Sequential(
            nn.Linear(self.config.attn_dim * 2, self.config.classifier_hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(self.config.classifier_hidden_dim // 2, self.config.image_dim),
        )
        self.domain_head = nn.Sequential(
            nn.Linear(self.config.attn_dim * 2, self.config.domain_hidden_dim),
            nn.ReLU(),
            nn.Dropout(self.config.domain_dropout),
            nn.Linear(self.config.domain_hidden_dim, 1),
        )

    def _to_tokens(self, feat):
        # Match MVCAF_backbone_test_v1.1 token handling so transformer backbones
        # can stay in token space instead of being forced through a CNN-only path.
        if feat.ndim == 4:
            if feat.shape[1] == self.feature_dim:
                feat = self.cnn_proj(feat)
                return feat.flatten(2).transpose(1, 2)
            if feat.shape[-1] == self.feature_dim:
                feat = feat.permute(0, 3, 1, 2).contiguous()
                feat = self.cnn_proj(feat)
                return feat.flatten(2).transpose(1, 2)
        if feat.ndim == 3:
            if feat.shape[-1] == self.feature_dim:
                return self.token_proj(feat)
            if feat.shape[1] == self.feature_dim:
                return self.token_proj(feat.transpose(1, 2))
        raise ValueError(f"Unsupported feature shape: {tuple(feat.shape)} for backbone={self.config.backbone_name}")

    def extract_features(self, views):
        f1 = self.backbone.forward_features(views[0])
        f2 = self.backbone.forward_features(views[1])
        t1 = self._to_tokens(f1)
        t2 = self._to_tokens(f2)
        for blk12, blk21 in zip(self.cross_12, self.cross_21):
            t1_prev, t2_prev = t1, t2
            t1 = blk12(t1_prev, t2_prev)
            t2 = blk21(t2_prev, t1_prev)
        return torch.cat([t1.mean(dim=1), t2.mean(dim=1)], dim=1)

    def forward(self, views, lambda_=0.0):
        feat = self.extract_features(views)
        out = {
            'class_logit': self.classifier(feat).view(-1),
            'physics_pred': self.physics_head(feat),
            'image_pred': self.image_head(feat),
            'feat': feat,
        }
        dom_feat = grad_reverse(feat, lambda_)
        out['domain_logit'] = self.domain_head(dom_feat).view(-1)
        return out


In [5]:

def mixup_multiview_batch(views, labels, alpha=0.2):
    if alpha <= 0:
        return views, labels, labels, 1.0
    lam = np.random.beta(alpha, alpha)
    batch_size = labels.size(0)
    index = torch.randperm(batch_size, device=labels.device)
    mixed_views = [lam * v + (1.0 - lam) * v[index, :] for v in views]
    return mixed_views, labels, labels[index], lam


def compute_lambda(epoch_idx, step_idx, steps_per_epoch, total_epochs, warmup_epochs=0, max_lambda=1.0, gamma=10.0):
    total_steps = max(1, total_epochs * steps_per_epoch)
    current_step = max(0, (epoch_idx - 1) * steps_per_epoch + step_idx)
    progress = current_step / total_steps
    warmup_progress = warmup_epochs / max(1, total_epochs)
    if progress <= warmup_progress:
        return 0.0
    effective_progress = (progress - warmup_progress) / max(1e-8, 1.0 - warmup_progress)
    lambda_base = 2.0 / (1.0 + np.exp(-gamma * effective_progress)) - 1.0
    return float(max_lambda * lambda_base)


def masked_smooth_l1(pred, target):
    mask = torch.isfinite(target).all(dim=1)
    if mask.any():
        return F.smooth_l1_loss(pred[mask], target[mask])
    return pred.sum() * 0.0


def train_one_epoch(model, train_loader, dev_loader, optimizer, device, epoch_idx, total_epochs, ema=None):
    model.train()
    dev_iter = cycle(dev_loader)
    total_rows = []
    steps = len(train_loader)
    for step_idx, batch in enumerate(tqdm(train_loader, desc='Training', dynamic_ncols=True, ascii=True), start=1):
        dev_batch = next(dev_iter)
        train_views = [v.to(device) for v in batch['views']]
        train_labels = batch['label'].to(device).float()
        train_phys = batch['physics_target'].to(device)
        train_img = batch['image_target'].to(device)
        dev_views = [v.to(device) for v in dev_batch['views']]

        optimizer.zero_grad()

        if CFG['MIXUP_ALPHA'] > 0 and np.random.rand() < CFG['MIXUP_PROB']:
            mixed_views, labels_a, labels_b, lam = mixup_multiview_batch(train_views, train_labels, CFG['MIXUP_ALPHA'])
            outputs = model(mixed_views, lambda_=0.0)
            loss_cls = lam * F.binary_cross_entropy_with_logits(outputs['class_logit'], labels_a) + (1.0 - lam) * F.binary_cross_entropy_with_logits(outputs['class_logit'], labels_b)
            loss_phys = outputs['physics_pred'].sum() * 0.0
            loss_img = outputs['image_pred'].sum() * 0.0
        else:
            outputs = model(train_views, lambda_=0.0)
            loss_cls = F.binary_cross_entropy_with_logits(outputs['class_logit'], train_labels)
            loss_phys = masked_smooth_l1(outputs['physics_pred'], train_phys)
            loss_img = masked_smooth_l1(outputs['image_pred'], train_img)

        domain_views = [torch.cat([tv, dv], dim=0) for tv, dv in zip(train_views, dev_views)]
        domain_labels = torch.cat([
            torch.zeros(train_views[0].size(0), device=device),
            torch.ones(dev_views[0].size(0), device=device),
        ], dim=0)
        grl_lambda = compute_lambda(
            epoch_idx,
            step_idx - 1,
            steps,
            total_epochs,
            warmup_epochs=CFG['GRL_WARMUP_EPOCHS'],
            max_lambda=CFG['GRL_MAX_LAMBDA'],
            gamma=CFG['GRL_GAMMA'],
        )
        domain_outputs = model(domain_views, lambda_=grl_lambda)
        loss_dom = F.binary_cross_entropy_with_logits(domain_outputs['domain_logit'], domain_labels)

        loss = loss_cls + CFG['PHYSICS_LOSS_WEIGHT'] * loss_phys + CFG['IMAGE_LOSS_WEIGHT'] * loss_img + CFG['DOMAIN_LOSS_WEIGHT'] * loss_dom
        loss.backward()
        optimizer.step()
        if ema is not None:
            ema.update(model)

        with torch.no_grad():
            domain_probs = torch.sigmoid(domain_outputs['domain_logit'])
            domain_acc = ((domain_probs > 0.5) == domain_labels).float().mean().item()

        total_rows.append({
            'loss': float(loss.item()),
            'loss_cls': float(loss_cls.item()),
            'loss_phys': float(loss_phys.item()),
            'loss_img': float(loss_img.item()),
            'loss_dom': float(loss_dom.item()),
            'domain_acc': float(domain_acc),
        })

    hist = pd.DataFrame(total_rows)
    return hist.mean().to_dict()


@torch.no_grad()
def evaluate_classification(model, loader, device):
    model.eval()
    all_probs = []
    all_labels = []
    physics_losses = []
    image_losses = []
    for batch in tqdm(loader, desc='Dev Eval', dynamic_ncols=True, ascii=True):
        views = [v.to(device) for v in batch['views']]
        labels = batch['label'].to(device).float()
        physics_target = batch['physics_target'].to(device)
        image_target = batch['image_target'].to(device)
        outputs = model(views, lambda_=0.0)
        probs = torch.sigmoid(outputs['class_logit'])
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        physics_losses.append(float(masked_smooth_l1(outputs['physics_pred'], physics_target).item()))
        image_losses.append(float(masked_smooth_l1(outputs['image_pred'], image_target).item()))

    all_probs = np.array(all_probs, dtype=np.float64)
    all_labels = np.array(all_labels, dtype=np.float64)
    p = np.clip(all_probs, 1e-15, 1 - 1e-15)
    logloss = float(-np.mean(all_labels * np.log(p) + (1.0 - all_labels) * np.log(1.0 - p)))
    acc = float(np.mean((all_probs > 0.5) == all_labels))
    auc = float(__import__('sklearn.metrics').metrics.roc_auc_score(all_labels, all_probs))
    return {
        'dev_logloss': logloss,
        'dev_acc': acc,
        'dev_auc': auc,
        'dev_phys_loss': float(np.mean(physics_losses)),
        'dev_img_loss': float(np.mean(image_losses)),
    }


@torch.no_grad()
def predict_test_probs(model, loader, device):
    model.eval()
    probs_all = []
    ids = []
    for batch in tqdm(loader, desc='Test Inference', dynamic_ncols=True, ascii=True):
        views = [v.to(device) for v in batch['views']]
        outputs = model(views, lambda_=0.0)
        probs = torch.sigmoid(outputs['class_logit'])
        probs_all.extend(probs.cpu().numpy())
        ids.extend(batch['id'])
    return ids, np.array(probs_all, dtype=np.float64)


In [6]:

model = TeacherRegularizedMultiView(TeacherRegularizationConfig()).to(device)
optimizer = optim.Adam(model.parameters(), lr=CFG['LEARNING_RATE'], weight_decay=CFG['WEIGHT_DECAY'])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['EPOCHS'], eta_min=CFG['MIN_LR'])
ema = ModelEMA(model, EMAConfig(decay=CFG['EMA_DECAY']))

artifacts = allocate_output_paths(experiment_name='teacher_regularization', major_version='v2.0')
best_model_path = artifacts['weight_path']
submission_path = artifacts['submission_path']
history_path = (Path.cwd() / '../../outputs/eda_preprocessing/teacher_regularization_v2.0_history.csv').resolve()
history_path.parent.mkdir(parents=True, exist_ok=True)
print('Artifact version:', artifacts['version'])
print('best_model_path:', best_model_path)
print('submission_path:', submission_path)

best_logloss = float('inf')
best_epoch = -1
patience_left = CFG['EARLY_STOPPING_PATIENCE']
history = []

for epoch in range(1, CFG['EPOCHS'] + 1):
    train_metrics = train_one_epoch(model, train_loader, dev_domain_loader, optimizer, device, epoch, CFG['EPOCHS'], ema=ema)
    eval_model = ema.ema_model if CFG['EMA_USE_FOR_EVAL'] else model
    dev_metrics = evaluate_classification(eval_model, dev_eval_loader, device)

    improved = dev_metrics['dev_logloss'] < best_logloss
    if improved:
        best_logloss = dev_metrics['dev_logloss']
        best_epoch = epoch
        patience_left = CFG['EARLY_STOPPING_PATIENCE']
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'ema_state_dict': ema.ema_model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'cfg': CFG,
            **dev_metrics,
        }, best_model_path)
    else:
        patience_left -= 1

    scheduler.step()
    row = {
        'epoch': epoch,
        **{k: float(v) for k, v in train_metrics.items()},
        **dev_metrics,
        'lr': float(optimizer.param_groups[0]['lr']),
        'improved': bool(improved),
        'patience_left': int(patience_left),
    }
    history.append(row)
    print(row)
    if patience_left < 0:
        print('early stop:', epoch)
        break

history_df = pd.DataFrame(history)
history_df.to_csv(history_path, index=False)
print('saved history:', history_path)


Artifact version: v2.0.3
best_model_path: /media/hdd0/whyz/structure-stability/outputs/weights/teacher_regularization_v2.0.3.pt
submission_path: /media/hdd0/whyz/structure-stability/outputs/submissions/teacher_regularization_v2.0.3.csv


Dev Eval: 100%|##########| 7/7 [00:00<00:00, 11.16it/s]


{'epoch': 1, 'loss': 0.4124555529907663, 'loss_cls': 0.3421434844903489, 'loss_phys': 0.2750616689321605, 'loss_img': 0.29073277190377184, 'loss_dom': 0.5497141355212699, 'domain_acc': 0.7180961896764472, 'dev_logloss': 0.6754574909034418, 'dev_acc': 0.55, 'dev_auc': 0.8822115384615384, 'dev_phys_loss': 2.0794443573270525, 'dev_img_loss': 0.7206063781465802, 'lr': 0.0002991810233575568, 'improved': True, 'patience_left': 5}


Dev Eval: 100%|##########| 7/7 [00:00<00:00, 11.35it/s]


{'epoch': 2, 'loss': 0.2966551686775811, 'loss_cls': 0.23255544183577628, 'loss_phys': 0.2424121007739388, 'loss_img': 0.2693989677429001, 'loss_dom': 0.5007844957265448, 'domain_acc': 0.7466866147327931, 'dev_logloss': 0.5879038316337455, 'dev_acc': 0.78, 'dev_auc': 0.971153846153846, 'dev_phys_loss': 2.1009385074887956, 'dev_img_loss': 0.7074633155550275, 'lr': 0.0002967330663097039, 'improved': True, 'patience_left': 5}


...

{'epoch': 17, 'loss': 0.08835242897708048, 'loss_cls': 0.05680253127765683, 'loss_phys': 0.11590837230633429, 'loss_img': 0.06295403687932351, 'loss_dom': 0.3891815057301775, 'domain_acc': 0.818406473765982, 'dev_logloss': 0.10741236084687605, 'dev_acc': 0.94, 'dev_auc': 0.9959935897435896, 'dev_phys_loss': 2.038223283631461, 'dev_img_loss': 0.5652817445141929, 'lr': 0.00011941720222274492, 'improved': False, 'patience_left': 1}



Dev Eval: 100%|##########| 7/7 [00:00<00:00, 11.12it/s]

{'epoch': 18, 'loss': 0.08726692996285063, 'loss_cls': 0.054645586391648356, 'loss_phys': 0.11034372454989662, 'loss_img': 0.05909268098680223, 'loss_dom': 0.42389776454644, 'domain_acc': 0.8061613491240968, 'dev_logloss': 0.10400314468160493, 'dev_acc': 0.93, 'dev_auc': 0.9967948717948718, 'dev_phys_loss': 2.0290781429835727, 'dev_img_loss': 0.5644107290676662, 'lr': 0.00010430195934094534, 'improved': False, 'patience_left': 0}



Dev Eval: 100%|##########| 7/7 [00:00<00:00, 11.12it/s]

{'epoch': 19, 'loss': 0.07160524453254456, 'loss_cls': 0.04231368795661454, 'loss_phys': 0.10653013191706046, 'loss_img': 0.04728287737352149, 'loss_dom': 0.3847352420078947, 'domain_acc': 0.8183289033935425, 'dev_logloss': 0.10504787304080124, 'dev_acc': 0.93, 'dev_auc': 0.9967948717948718, 'dev_phys_loss': 2.0242726802825928, 'dev_img_loss': 0.5632450367723193, 'lr': 8.969287186016786e-05, 'improved': False, 'patience_left': -1}
early stop: 19
saved history: /media/hdd0/whyz/structure-stability/outputs/eda_preprocessing/teacher_regularization_v2.0_history.csv


In [ ]:

checkpoint = torch.load(best_model_path, map_location=device, weights_only=False)
best_state = checkpoint['ema_state_dict'] if CFG['EMA_USE_FOR_EVAL'] else checkpoint['model_state_dict']
model.load_state_dict(best_state)
final_dev_metrics = evaluate_classification(model, dev_eval_loader, device)
print('best_epoch:', best_epoch)
print(final_dev_metrics)

ids, test_probs = predict_test_probs(model, test_loader, device)
submission = pd.DataFrame({
    'id': ids,
    'unstable_prob': test_probs,
    'stable_prob': 1.0 - test_probs,
})
submission.to_csv(submission_path, index=False, encoding='utf-8-sig')
print('saved submission:', submission_path)
submission.head()


Dev Eval: 100%|##########| 7/7 [00:00<00:00, 10.77it/s]


best_epoch: 13
{'dev_logloss': 0.10134854838949878, 'dev_acc': 0.96, 'dev_auc': 0.9951923076923077, 'dev_phys_loss': 2.055924126080104, 'dev_img_loss': 0.5768773938928332}


Test Inference: 100%|##########| 63/63 [00:03<00:00, 19.85it/s]

saved submission: /media/hdd0/whyz/structure-stability/outputs/submissions/teacher_regularization_v2.0.3.csv


,id,unstable_prob,stable_prob
0,TEST_0001,0.010687,0.989313
1,TEST_0002,0.994519,0.005481
2,TEST_0003,0.993127,0.006873
3,TEST_0004,0.988921,0.011079
4,TEST_0005,0.003758,0.996242


: 